In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.etl.context import PipelineContext
from src.etl.run_tracker import PipelineRun
from src.etl import extract, transform, enrich, load

14:55:18 | INFO    | etl          | Logging to: C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\logs\etl_20260511_145518.log


In [2]:
import pandas as pd
from src.warehouse import get_engine

engine = get_engine()

print("=" * 60)
print("Pre-test warehouse state")
print("=" * 60)

with engine.begin() as conn:
    counts = pd.read_sql("""
        SELECT 'dim_asset' AS table_name, COUNT(*) AS rows FROM warehouse.dim_asset
        UNION ALL SELECT 'dim_port', COUNT(*) FROM warehouse.dim_port
        UNION ALL SELECT 'dim_attack_type', COUNT(*) FROM warehouse.dim_attack_type
        UNION ALL SELECT 'dim_protocol', COUNT(*) FROM warehouse.dim_protocol
        UNION ALL SELECT 'fact_security_event', COUNT(*) FROM warehouse.fact_security_event
        ORDER BY table_name
    """, conn)
print(counts.to_string(index=False))

Pre-test warehouse state
         table_name  rows
          dim_asset   743
    dim_attack_type    16
           dim_port 10078
       dim_protocol    12
fact_security_event     0


In [3]:
print("=" * 60)
print("Test 1: Sample mode — full pipeline incl. LOAD")
print("=" * 60)

ctx = PipelineContext(run_mode="sample", sample_size=10_000)

with PipelineRun(ctx) as run:
    with run.stage("extract") as s:
        extract.run(ctx)
        s.set_metrics(rows_in=0, rows_out=ctx.rows_extracted)

    with run.stage("transform") as s:
        transform.run(ctx)
        s.set_metrics(rows_in=ctx.rows_extracted, rows_out=ctx.rows_transformed)

    with run.stage("enrich") as s:
        enrich.run(ctx)
        s.set_metrics(rows_in=ctx.rows_transformed, rows_out=ctx.rows_enriched)

    with run.stage("load") as s:
        loaded = load.run(ctx)
        s.set_metrics(rows_in=ctx.rows_transformed, rows_out=loaded)

print(f"\n✅ Sample mode complete — {ctx.rows_loaded:,} rows loaded")

Test 1: Sample mode — full pipeline incl. LOAD
14:55:20 | INFO    | etl.tracker  | ╔══ Pipeline run 15 started: cicids_to_warehouse (sample) ══╗
14:55:20 | INFO    | etl.tracker  | ▶ Starting stage: extract
14:55:20 | INFO    | etl.extract  | Reading stratified sample (n=10,000) from cicids_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

14:55:37 | INFO    | etl.extract  | Loaded 8,047 rows × 87 columns into memory
14:55:37 | WARNING | etl.extract  | Dropped 1,000 invalid rows (missing timestamp or IPs)
14:55:37 | INFO    | etl.extract  | Attack family distribution (top): {'DDoS': 1000, 'DoS': 1000, 'Benign': 1000, 'Botnet': 1000, 'Brute Force': 1000, 'Reconnaissance': 1000, 'Web Attack': 1000, 'Infiltration': 36, 'Exploit': 11}
14:55:37 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 17.3s (rows: 0 → 7,047)
14:55:37 | INFO    | etl.tracker  | ▶ Starting stage: transform
14:55:37 | INFO    | etl.transform | Transforming 7,047 rows
14:55:37 | INFO    | etl.transform | Deriving date_sk, time_sk, is_after_hours, is_weekend
14:55:37 | INFO    | etl.transform | Classifying source IPs
14:55:37 | INFO    | etl.transform | Classifying destination IPs
14:55:37 | INFO    | etl.transform | Source IP classification: {'internal': 6507, 'external': 540}
14:55:37 | INFO    | etl.transform | Destination IP classification: {'

In [4]:
print("=" * 60)
print("Post-load warehouse state")
print("=" * 60)

with engine.begin() as conn:
    counts = pd.read_sql("""
        SELECT 'dim_asset' AS table_name, COUNT(*) AS rows FROM warehouse.dim_asset
        UNION ALL SELECT 'dim_port', COUNT(*) FROM warehouse.dim_port
        UNION ALL SELECT 'fact_security_event', COUNT(*) FROM warehouse.fact_security_event
        ORDER BY table_name
    """, conn)
print(counts.to_string(index=False))

print("\n=== dim_asset sample (first 5 internal + first 5 external) ===")
with engine.begin() as conn:
    internal = pd.read_sql("""
        SELECT asset_identifier, is_internal, country_iso, country_name, asset_type
        FROM warehouse.dim_asset
        WHERE is_internal = 1
        ORDER BY asset_sk
        LIMIT 5
    """, conn)
    external = pd.read_sql("""
        SELECT asset_identifier, is_internal, country_iso, country_name, abuse_confidence
        FROM warehouse.dim_asset
        WHERE is_internal = 0
        ORDER BY asset_sk
        LIMIT 5
    """, conn)
print("Internal:")
print(internal.to_string(index=False))
print("\nExternal (with geo enrichment):")
print(external.to_string(index=False))

print("\n=== Fact table sample (5 rows showing FK joins work) ===")
with engine.begin() as conn:
    sample = pd.read_sql("""
        SELECT 
            f.event_time,
            sa.asset_identifier AS src_ip,
            da.asset_identifier AS dest_ip,
            f.src_port_sk, f.dest_port_sk,
            p.protocol_name,
            at.attack_label,
            f.attack_family_denorm,
            f.is_attack,
            f.flow_duration,
            f.total_fwd_packets
        FROM warehouse.fact_security_event f
        LEFT JOIN warehouse.dim_asset sa ON f.src_asset_sk = sa.asset_sk
        LEFT JOIN warehouse.dim_asset da ON f.dest_asset_sk = da.asset_sk
        LEFT JOIN warehouse.dim_protocol p ON f.protocol_sk = p.protocol_sk
        LEFT JOIN warehouse.dim_attack_type at ON f.attack_sk = at.attack_sk
        LIMIT 5
    """, conn)
print(sample.to_string(index=False))

Post-load warehouse state
         table_name  rows
          dim_asset  1051
           dim_port 12895
fact_security_event  7047

=== dim_asset sample (first 5 internal + first 5 external) ===
Internal:
asset_identifier  is_internal country_iso country_name asset_type
   192.168.10.12            1        None         None       host
   192.168.10.50            1        None         None       host
   192.168.10.16            1        None         None       host
    192.168.10.9            1        None         None       host
   192.168.10.15            1        None         None       host

External (with geo enrichment):
asset_identifier  is_internal country_iso  country_name  abuse_confidence
  205.185.216.42            0          US United States               0.0
     23.111.9.30            0          US United States               0.0
    52.26.121.92            0          US United States               NaN
     72.21.91.29            0          US United States               0

In [5]:
print("=" * 60)
print("Data quality spot checks")
print("=" * 60)

with engine.begin() as conn:
    # Check 1: Attack family distribution should match what we loaded
    print("\n=== Attack family distribution in fact table ===")
    dist = pd.read_sql("""
        SELECT attack_family_denorm, COUNT(*) AS events
        FROM warehouse.fact_security_event
        GROUP BY attack_family_denorm
        ORDER BY events DESC
    """, conn)
    print(dist.to_string(index=False))

    # Check 2: FK integrity — no orphaned references
    print("\n=== FK resolution rates ===")
    fk_check = pd.read_sql("""
        SELECT 
            COUNT(*) AS total_rows,
            COUNT(src_asset_sk) AS has_src_asset,
            COUNT(dest_asset_sk) AS has_dest_asset,
            COUNT(src_port_sk) AS has_src_port,
            COUNT(dest_port_sk) AS has_dest_port,
            COUNT(attack_sk) AS has_attack,
            COUNT(protocol_sk) AS has_protocol
        FROM warehouse.fact_security_event
    """, conn)
    print(fk_check.to_string(index=False))

    # Check 3: External IP enrichment quality
    print("\n=== External IP geographic enrichment ===")
    geo_check = pd.read_sql("""
        SELECT 
            COUNT(*) AS external_ips,
            COUNT(country_iso) AS with_country,
            COUNT(latitude) AS with_coords,
            COUNT(CASE WHEN abuse_confidence IS NOT NULL THEN 1 END) AS with_reputation
        FROM warehouse.dim_asset
        WHERE is_internal = 0
    """, conn)
    print(geo_check.to_string(index=False))

    # Check 4: Date_sk → dim_date FK integrity (no orphans)
    print("\n=== Date join integrity ===")
    date_check = pd.read_sql("""
        SELECT 
            COUNT(*) AS fact_rows,
            COUNT(DISTINCT f.date_sk) AS distinct_dates,
            COUNT(d.full_date) AS resolved_dates,
            MIN(d.full_date) AS earliest,
            MAX(d.full_date) AS latest
        FROM warehouse.fact_security_event f
        LEFT JOIN warehouse.dim_date d ON f.date_sk = d.date_sk
    """, conn)
    print(date_check.to_string(index=False))

Data quality spot checks

=== Attack family distribution in fact table ===
attack_family_denorm  events
          Web Attack    1000
              Botnet    1000
         Brute Force    1000
                DDoS    1000
                 DoS    1000
      Reconnaissance    1000
              Benign    1000
        Infiltration      36
             Exploit      11

=== FK resolution rates ===
 total_rows  has_src_asset  has_dest_asset  has_src_port  has_dest_port  has_attack  has_protocol
       7047           7047            7047          7047           7047        7047          7047

=== External IP geographic enrichment ===
 external_ips  with_country  with_coords  with_reputation
         1035          1004         1004              185

=== Date join integrity ===
 fact_rows  distinct_dates  resolved_dates   earliest     latest
      7047               5            7047 2017-07-03 2017-07-07


In [6]:
import time

print("=" * 60)
print("Test 2: FULL MODE — Loading all 2.83M rows to PostgreSQL")
print("=" * 60)
print("\n⏱️  Estimated 3-8 minutes total:")
print("    - Extract:    ~45s (re-reads parquet)")
print("    - Transform:  ~25s")
print("    - Enrich:     ~30s (cached from earlier runs)")
print("    - Load:       ~3-5 min (dim upserts + 2.83M COPY)")
print()

start = time.time()

ctx = PipelineContext(run_mode="full")

with PipelineRun(ctx) as run:
    with run.stage("extract") as s:
        extract.run(ctx)
        s.set_metrics(rows_in=0, rows_out=ctx.rows_extracted)

    with run.stage("transform") as s:
        transform.run(ctx)
        s.set_metrics(rows_in=ctx.rows_extracted, rows_out=ctx.rows_transformed)

    with run.stage("enrich") as s:
        enrich.run(ctx)
        s.set_metrics(rows_in=ctx.rows_transformed, rows_out=ctx.rows_enriched)

    with run.stage("load") as s:
        loaded = load.run(ctx)
        s.set_metrics(rows_in=ctx.rows_transformed, rows_out=loaded)

total_time = time.time() - start
print(f"\n✅ Full mode complete — {ctx.rows_loaded:,} rows loaded in {total_time:.1f}s")
print(f"   End-to-end throughput: {ctx.rows_loaded / total_time:,.0f} rows/sec")

Test 2: FULL MODE — Loading all 2.83M rows to PostgreSQL

⏱️  Estimated 3-8 minutes total:
    - Extract:    ~45s (re-reads parquet)
    - Transform:  ~25s
    - Enrich:     ~30s (cached from earlier runs)
    - Load:       ~3-5 min (dim upserts + 2.83M COPY)

15:05:15 | INFO    | etl.tracker  | ╔══ Pipeline run 16 started: cicids_to_warehouse (full) ══╗
15:05:15 | INFO    | etl.tracker  | ▶ Starting stage: extract
15:05:15 | INFO    | etl.extract  | Reading cicids_clean.parquet (293.0 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15:05:42 | INFO    | etl.extract  | Loaded 3,119,345 rows × 87 columns into memory
15:05:53 | WARNING | etl.extract  | Dropped 288,602 invalid rows (missing timestamp or IPs)
15:05:54 | INFO    | etl.extract  | Attack family distribution (top): {'Benign': 2273097, 'DoS': 252661, 'Reconnaissance': 158930, 'DDoS': 128027, 'Brute Force': 13835, 'Web Attack': 2180, 'Botnet': 1966, 'Infiltration': 36, 'Exploit': 11}
15:05:54 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 39.1s (rows: 0 → 2,830,743)
15:05:54 | INFO    | etl.tracker  | ▶ Starting stage: transform
15:05:54 | INFO    | etl.transform | Transforming 2,830,743 rows
15:05:55 | INFO    | etl.transform | Deriving date_sk, time_sk, is_after_hours, is_weekend
15:05:56 | INFO    | etl.transform | Classifying source IPs
15:05:58 | INFO    | etl.transform | Classifying destination IPs
15:06:00 | INFO    | etl.transform | Source IP classification: {'internal': 2467175, 'external': 363568}
15:06:00 | INFO    | etl.transform | Des

In [7]:
import pandas as pd
from src.warehouse import get_engine

engine = get_engine()

print("=" * 70)
print("FULL LOAD VALIDATION")
print("=" * 70)

with engine.begin() as conn:
    # Row counts across all warehouse tables
    print("\n=== Warehouse table counts ===")
    counts = pd.read_sql("""
        SELECT 'dim_date' AS table_name, COUNT(*) AS rows FROM warehouse.dim_date
        UNION ALL SELECT 'dim_time', COUNT(*) FROM warehouse.dim_time
        UNION ALL SELECT 'dim_asset', COUNT(*) FROM warehouse.dim_asset
        UNION ALL SELECT 'dim_port', COUNT(*) FROM warehouse.dim_port
        UNION ALL SELECT 'dim_protocol', COUNT(*) FROM warehouse.dim_protocol
        UNION ALL SELECT 'dim_attack_type', COUNT(*) FROM warehouse.dim_attack_type
        UNION ALL SELECT 'dim_activity_type', COUNT(*) FROM warehouse.dim_activity_type
        UNION ALL SELECT 'fact_security_event', COUNT(*) FROM warehouse.fact_security_event
        ORDER BY table_name
    """, conn)
    print(counts.to_string(index=False))

    # Attack family distribution (CRITICAL — must match Week 1 EDA)
    print("\n=== Attack family distribution ===")
    dist = pd.read_sql("""
        SELECT 
            attack_family_denorm AS family,
            COUNT(*) AS events,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM warehouse.fact_security_event
        GROUP BY attack_family_denorm
        ORDER BY events DESC
    """, conn)
    print(dist.to_string(index=False))

    # Per-day events (CRITICAL — must match the date_sk distribution from Day 3)
    print("\n=== Events per day ===")
    daily = pd.read_sql("""
        SELECT d.full_date, d.day_name, COUNT(*) AS events
        FROM warehouse.fact_security_event f
        JOIN warehouse.dim_date d ON f.date_sk = d.date_sk
        GROUP BY d.full_date, d.day_name
        ORDER BY d.full_date
    """, conn)
    print(daily.to_string(index=False))

FULL LOAD VALIDATION

=== Warehouse table counts ===
         table_name    rows
  dim_activity_type       7
          dim_asset   19129
    dim_attack_type      16
           dim_date    2922
           dim_port   64655
       dim_protocol      12
           dim_time    1440
fact_security_event 2830743

=== Attack family distribution ===
        family  events   pct
        Benign 2273097 80.30
           DoS  252661  8.93
Reconnaissance  158930  5.61
          DDoS  128027  4.52
   Brute Force   13835  0.49
    Web Attack    2180  0.08
        Botnet    1966  0.07
  Infiltration      36  0.00
       Exploit      11  0.00

=== Events per day ===
 full_date  day_name  events
2017-07-03    Monday  529918
2017-07-04   Tuesday  445909
2017-07-05 Wednesday  692703
2017-07-06  Thursday  458968
2017-07-07    Friday  703245


In [8]:
print("=" * 70)
print("ATTACK WINDOW VALIDATION (against CICIDS2017 documentation)")
print("=" * 70)

with engine.begin() as conn:
    # Document the well-known CICIDS2017 attacks and their expected times
    attack_windows = pd.read_sql("""
        SELECT 
            at.attack_label,
            at.attack_family,
            d.full_date,
            MIN(f.event_time)::TIME AS first_seen,
            MAX(f.event_time)::TIME AS last_seen,
            COUNT(*) AS events
        FROM warehouse.fact_security_event f
        JOIN warehouse.dim_attack_type at ON f.attack_sk = at.attack_sk
        JOIN warehouse.dim_date d ON f.date_sk = d.date_sk
        WHERE at.attack_label NOT IN ('BENIGN', 'Unlabeled')
        GROUP BY at.attack_label, at.attack_family, d.full_date
        ORDER BY d.full_date, MIN(f.event_time)
    """, conn)
    print(attack_windows.to_string(index=False))

    print("\n=== Heartbleed precision check (documented: Wed 15:12-15:32) ===")
    heartbleed = pd.read_sql("""
        SELECT 
            f.event_time,
            sa.asset_identifier AS src_ip,
            da.asset_identifier AS dest_ip,
            f.flow_duration
        FROM warehouse.fact_security_event f
        JOIN warehouse.dim_asset sa ON f.src_asset_sk = sa.asset_sk
        JOIN warehouse.dim_asset da ON f.dest_asset_sk = da.asset_sk
        JOIN warehouse.dim_attack_type at ON f.attack_sk = at.attack_sk
        WHERE at.attack_label = 'Heartbleed'
        ORDER BY f.event_time
    """, conn)
    print(heartbleed.to_string(index=False))

ATTACK WINDOW VALIDATION (against CICIDS2017 documentation)
              attack_label  attack_family  full_date first_seen last_seen  events
               FTP-Patator    Brute Force 2017-07-04   09:17:00  10:30:00    7938
               SSH-Patator    Brute Force 2017-07-04   14:09:00  15:11:00    5897
             DoS slowloris            DoS 2017-07-05   09:01:00  14:25:00    5796
          DoS Slowhttptest            DoS 2017-07-05   10:15:00  10:37:00    5499
                  DoS Hulk            DoS 2017-07-05   10:43:00  11:07:00  231073
             DoS GoldenEye            DoS 2017-07-05   11:10:00  11:19:00   10293
                Heartbleed        Exploit 2017-07-05   15:12:00  15:32:00      11
  Web Attack - Brute Force     Web Attack 2017-07-06   09:15:00  10:00:00    1507
          Web Attack - XSS     Web Attack 2017-07-06   10:15:00  10:35:00     652
Web Attack - Sql Injection     Web Attack 2017-07-06   10:40:00  10:42:00      21
              Infiltration   Infiltrat

In [9]:
print("=" * 70)
print("GEOGRAPHIC ENRICHMENT QUALITY")
print("=" * 70)

with engine.begin() as conn:
    print("\n=== External IP geo coverage ===")
    coverage = pd.read_sql("""
        SELECT 
            COUNT(*) AS external_ips,
            COUNT(country_iso) AS with_country,
            COUNT(latitude) AS with_coords,
            COUNT(CASE WHEN abuse_confidence IS NOT NULL THEN 1 END) AS with_reputation,
            ROUND(100.0 * COUNT(country_iso) / COUNT(*), 2) AS pct_geo,
            ROUND(100.0 * COUNT(CASE WHEN abuse_confidence IS NOT NULL THEN 1 END) / COUNT(*), 2) AS pct_rep
        FROM warehouse.dim_asset
        WHERE is_internal = 0
    """, conn)
    print(coverage.to_string(index=False))

    print("\n=== Top 15 countries (by IP count) ===")
    countries = pd.read_sql("""
        SELECT 
            country_iso,
            country_name,
            COUNT(*) AS distinct_ips
        FROM warehouse.dim_asset
        WHERE is_internal = 0
          AND country_iso IS NOT NULL
        GROUP BY country_iso, country_name
        ORDER BY distinct_ips DESC
        LIMIT 15
    """, conn)
    print(countries.to_string(index=False))

    print("\n=== Documented attacker IP detail ===")
    attacker = pd.read_sql("""
        SELECT 
            asset_identifier, country_iso, country_name, city,
            latitude, longitude, abuse_confidence, is_known_attacker
        FROM warehouse.dim_asset
        WHERE asset_identifier = '205.174.165.73'
    """, conn)
    print(attacker.to_string(index=False))

GEOGRAPHIC ENRICHMENT QUALITY

=== External IP geo coverage ===
 external_ips  with_country  with_coords  with_reputation  pct_geo  pct_rep
        19040         18520        18520                0    97.27      0.0

=== Top 15 countries (by IP count) ===
country_iso    country_name  distinct_ips
         US   United States         13354
         JP           Japan           662
         DE         Germany           542
         CN           China           475
         IE         Ireland           416
         RU          Russia           376
         CA          Canada           323
         FR          France           285
         NL The Netherlands           262
         KR     South Korea           179
         BR          Brazil           170
         SG       Singapore           151
         GB  United Kingdom           133
         NZ     New Zealand           127
         PL          Poland           119

=== Documented attacker IP detail ===
asset_identifier country_iso coun

In [10]:
import time

print("=" * 70)
print("DASHBOARD QUERY PERFORMANCE TEST")
print("=" * 70)

with engine.begin() as conn:
    print("\n=== Hourly attack heatmap (using analytical view) ===")
    t0 = time.time()
    heatmap = pd.read_sql("""
        SELECT * FROM warehouse.v_hourly_threat_pattern
        ORDER BY day_name, hour_24, attack_family
        LIMIT 20
    """, conn)
    print(f"Query time: {time.time()-t0:.2f}s")
    print(heatmap.to_string(index=False))

    print("\n=== Top attackers view performance ===")
    t0 = time.time()
    attackers = pd.read_sql("""
        SELECT attacker_ip, country_iso, country_name, attack_count, unique_targets
        FROM warehouse.v_top_attackers
        ORDER BY attack_count DESC
        LIMIT 10
    """, conn)
    print(f"Query time: {time.time()-t0:.2f}s")
    print(attackers.to_string(index=False))

DASHBOARD QUERY PERFORMANCE TEST

=== Hourly attack heatmap (using analytical view) ===
Query time: 2.01s
 day_name  hour_24  day_part  attack_family  event_count
   Friday        9   morning         Botnet            2
   Friday       10   morning         Botnet         1448
   Friday       11   morning         Botnet          264
   Friday       12 afternoon         Botnet          252
   Friday       13 afternoon Reconnaissance          185
   Friday       14 afternoon Reconnaissance       138680
   Friday       15 afternoon           DDoS        23865
   Friday       15 afternoon Reconnaissance        20065
   Friday       16 afternoon           DDoS       104162
 Thursday        9   morning     Web Attack         1500
 Thursday       10   morning     Web Attack          680
 Thursday       14 afternoon   Infiltration           18
 Thursday       15 afternoon   Infiltration           18
  Tuesday        9   morning    Brute Force         5266
  Tuesday       10   morning    Brute F